In [ ]:
%pip uninstall --yes keras matplotlib scikit-learn tensorflow

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    wheels_dir = f'{temp_dir}/wheels'
    os.makedirs(temp_dir, exist_ok=True)

    if not os.path.exists(input_archive):
        raise FileNotFoundError(f'Archive not found: {input_archive}')

    # Extract when wheels folder is missing or empty.
    if (not os.path.isdir(wheels_dir)) or (not os.listdir(wheels_dir)):
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)

    if (not os.path.isdir(wheels_dir)) or (not os.listdir(wheels_dir)):
        raise RuntimeError(f'Wheels folder missing after extraction: {wheels_dir}')

    subprocess.run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--find-links',
        wheels_dir,
        'unsloth',
        'trl',
        'vllm',
        'openai_harmony'
    ], check=True)


In [ ]:
set_env(
    input_archive='/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz',
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
import os

# Runtime environment settings for vLLM / tokenizer behavior in Kaggle.
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'
os.environ['VLLM_ATTENTION_BACKEND'] = 'TRITON_ATTN'


In [ ]:
import gc
import io
import os
import re
import sys
import ast
import csv
import json
import math
import time
import queue
import threading
import contextlib
import subprocess
import socket
from dataclasses import dataclass
from typing import Optional

from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import requests
import pandas as pd
import polars as pl

from openai import OpenAI
from jupyter_client import KernelManager
from transformers import set_seed

try:
    import kaggle_evaluation.aimo_3_inference_server as aimo3_inference
except Exception:
    aimo3_inference = None


In [ ]:
@dataclass
class CFG:
    system_prompt: str = (
        'You are an IMO-style math solver. '
        'The final answer must be an integer in [0, 99999] and formatted as \boxed{n}. '
        'Use the python_math tool whenever calculation helps.'
    )
    preference_prompt: str = (
        'There is access to Python with math, numpy and sympy. '
        'Show the final answer only as \boxed{n}.'
    )
    tool_prompt: str = 'Use this tool to execute Python code for calculations. Use print(...) to show output.'

    served_model_name: str = 'qwq-32b'
    model_path: str = '/kaggle/input/models/qwen-lm/qwq-32b/transformers/qwq-32b/1'

    dtype: str = 'bfloat16'
    kv_cache_dtype: str = 'auto'
    gpu_memory_utilization: float = 0.9

    notebook_limit: int = 17400
    high_problem_timeout: int = 600
    base_problem_timeout: int = 180

    server_timeout: int = 180
    session_timeout: int = 960
    request_timeout: int = 960
    jupyter_timeout: int = 8
    sandbox_timeout: int = 5

    context_tokens: int = 32768
    buffer_tokens: int = 512
    stream_interval: int = 200
    batch_size: int = 8

    search_tokens: int = 32
    top_logprobs: int = 5
    attempts: int = 4
    workers: int = 4
    turns: int = 16
    early_stop: int = 2

    max_token_schedule: tuple = (256, 160, 96)

    seed: int = 2570
    temperature: float = 0.2
    min_p: float = 0.02


def resolve_model_context_tokens(cfg: CFG) -> int:
    config_path = os.path.join(cfg.model_path, 'config.json')
    detected = None

    try:
        with open(config_path, 'r', encoding='utf-8') as f:
            model_cfg = json.load(f)

        for key in ('max_position_embeddings', 'model_max_length', 'max_seq_len', 'seq_length'):
            value = model_cfg.get(key)
            if isinstance(value, int) and value > 0:
                detected = value
                break
    except Exception as exc:
        print(f'Could not read model config for context length: {exc}')

    if detected is None:
        detected = cfg.context_tokens

    # Guardrail for stable vLLM memory usage during multi-attempt solving.
    cfg.context_tokens = int(max(4096, min(detected, 32768)))
    return cfg.context_tokens


def safe_set_seed(seed: int):
    try:
        set_seed(seed)
        return
    except ModuleNotFoundError as exc:
        if 'tensorflow' not in str(exc).lower():
            raise

    import random
    random.seed(seed)

    try:
        import numpy as np
        np.random.seed(seed)
    except Exception:
        pass

    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


cfg = CFG()
resolved_context = resolve_model_context_tokens(cfg)
print(f'Configured model: {cfg.served_model_name}')
print(f'Model path: {cfg.model_path}')
print(f'Context tokens: {resolved_context}')
safe_set_seed(cfg.seed)


In [ ]:
class AIMO3Sandbox:
    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float):
        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None

        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback_lines: list[str]) -> str:
        clean_lines = []
        for frame in traceback_lines:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)
            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue
            clean_lines.append(clean_frame)
        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:
        client = self._client
        effective_timeout = timeout or self._default_timeout

        msg_id = client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout_parts = []
        stderr_parts = []
        start_time = time.time()

        while True:
            if time.time() - start_time > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')
                if content.get('name') == 'stdout':
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)
            elif msg_type == 'error':
                stderr_parts.append(self._format_error(content.get('traceback', [])))
            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')
                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')
            elif msg_type == 'status' and content.get('execution_state') == 'idle':
                break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr
        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def reset(self):
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def close(self):
        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()
        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def __del__(self):
        self.close()


class SandboxPool:
    def __init__(self, workers: int, timeout: float):
        self.workers = workers
        self.timeout = timeout
        self.pool = queue.Queue()

    def initialize(self):
        print(f'Initializing {self.workers} persistent Jupyter kernels...')
        start = time.time()

        def _create():
            return AIMO3Sandbox(timeout=self.timeout)

        with ThreadPoolExecutor(max_workers=self.workers) as ex:
            futures = [ex.submit(_create) for _ in range(self.workers)]
            for fut in as_completed(futures):
                self.pool.put(fut.result())

        print(f'Kernels initialized in {time.time() - start:.2f} seconds.\n')

    def acquire(self, timeout: float):
        return self.pool.get(timeout=timeout)

    def release(self, sandbox: AIMO3Sandbox):
        self.pool.put(sandbox)

    def close(self):
        while not self.pool.empty():
            with contextlib.suppress(Exception):
                sb = self.pool.get_nowait()
                sb.close()


In [ ]:
class VLLMService:
    @staticmethod
    def _find_free_port() -> int:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.bind(('127.0.0.1', 0))
            return int(sock.getsockname()[1])

    def __init__(self, cfg: CFG, port: int | None = None):
        self.cfg = cfg
        self.port = int(port) if port is not None else self._find_free_port()
        self.base_url = f'http://127.0.0.1:{self.port}/v1'
        self.log_file = None
        self.server_process = None
        self.server_mode = 'unknown'

    def preload_model_weights(self):
        if not os.path.exists(self.cfg.model_path):
            print(f'Model path not found: {self.cfg.model_path}. Skipping preload.')
            return

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start = time.time()

        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                p = os.path.join(root, file_name)
                if os.path.isfile(p):
                    files_to_load.append(p)
                    total_size += os.path.getsize(p)

        def _read_file(path: str):
            with open(path, 'rb') as f:
                while f.read(1024 * 1024 * 64):
                    pass

        if files_to_load:
            with ThreadPoolExecutor(max_workers=min(self.cfg.workers, 8)) as ex:
                list(ex.map(_read_file, files_to_load))

        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {time.time() - start:.2f} seconds.\n')

    def _build_cmd(self, advanced: bool) -> list[str]:
        cmd = [
            sys.executable,
            '-m',
            'vllm.entrypoints.openai.api_server',
            '--seed',
            str(self.cfg.seed),
            '--model',
            self.cfg.model_path,
            '--served-model-name',
            self.cfg.served_model_name,
            '--tensor-parallel-size',
            '1',
            '--max-num-seqs',
            str(self.cfg.batch_size),
            '--gpu-memory-utilization',
            str(self.cfg.gpu_memory_utilization),
            '--host',
            '0.0.0.0',
            '--port',
            str(self.port),
            '--dtype',
            self.cfg.dtype,
            '--max-model-len',
            str(self.cfg.context_tokens),
            '--stream-interval',
            str(self.cfg.stream_interval),
            '--enable-auto-tool-choice',
            '--tool-call-parser',
            'hermes',
            '--enforce-eager',
        ]

        if self.cfg.kv_cache_dtype != 'auto':
            cmd.extend(['--kv-cache-dtype', self.cfg.kv_cache_dtype])

        if advanced:
            cmd.extend(['--async-scheduling', '--disable-log-stats', '--enable-prefix-caching'])

        return cmd

    def _spawn(self, advanced: bool):
        log_name = 'vllm_server_advanced.log' if advanced else 'vllm_server_basic.log'
        log_file = open(log_name, 'w', encoding='utf-8')
        cmd = self._build_cmd(advanced)
        print('Starting vLLM with command:')
        print(' '.join(cmd))
        proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)
        return proc, log_file

    def _wait_http_ready(self, proc, log_file, timeout: int):
        start = time.time()
        for _ in range(timeout):
            if proc.poll() is not None:
                log_file.flush()
                with open(log_file.name, 'r', encoding='utf-8', errors='ignore') as f:
                    logs = f.read()
                raise RuntimeError(f'Server died with code {proc.returncode}. Logs:\n{logs}')

            try:
                r = requests.get(f'{self.base_url}/models', timeout=5)
                if r.status_code == 200:
                    print(f'Server is ready (took {time.time() - start:.2f} seconds).\n')
                    return
            except Exception:
                pass
            time.sleep(1)

        raise RuntimeError('Server failed to start within timeout.')

    def start(self):
        self.preload_model_weights()

        try:
            proc, log_file = self._spawn(advanced=True)
            self._wait_http_ready(proc, log_file, self.cfg.server_timeout)
            self.server_process, self.log_file, self.server_mode = proc, log_file, 'advanced'
            return
        except Exception as exc:
            print(f'Advanced server startup failed: {exc}')
            with contextlib.suppress(Exception):
                proc.terminate()
                proc.wait(timeout=10)
            with contextlib.suppress(Exception):
                log_file.close()

        proc, log_file = self._spawn(advanced=False)
        self._wait_http_ready(proc, log_file, self.cfg.server_timeout)
        self.server_process, self.log_file, self.server_mode = proc, log_file, 'basic'

    def close(self):
        if self.server_process is not None:
            with contextlib.suppress(Exception):
                self.server_process.terminate()
                self.server_process.wait(timeout=15)
        if self.log_file is not None:
            with contextlib.suppress(Exception):
                self.log_file.close()


In [ ]:
class AIMO3Solver:
    def __init__(self, cfg: CFG, port: int | None = None):
        self.cfg = cfg
        self.service = VLLMService(cfg, port=port)
        self.service.start()

        self.client = OpenAI(base_url=self.service.base_url, api_key='sk-local', timeout=cfg.session_timeout)
        self._wait_for_api()

        self.sandbox_pool = SandboxPool(workers=cfg.workers, timeout=cfg.jupyter_timeout)
        self.sandbox_pool.initialize()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50

        self.tools = [
            {
                'type': 'function',
                'function': {
                    'name': 'python_math',
                    'description': cfg.tool_prompt,
                    'parameters': {
                        'type': 'object',
                        'properties': {'code': {'type': 'string', 'description': 'Python code to run'}},
                        'required': ['code'],
                    },
                },
            }
        ]

    def _wait_for_api(self):
        print('Validating vLLM API availability...')
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if self.service.server_process.poll() is not None:
                raise RuntimeError('vLLM process exited before API became ready.')
            try:
                self.client.models.list()
                print(f'API check passed (took {time.time() - start:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('Server API validation timed out.')

    def _scan_for_answer(self, text: str) -> int | None:
        boxed = re.findall(r'\boxed\s*\{\s*([0-9,]+)\s*\}', text)
        if boxed:
            try:
                v = int(boxed[-1].replace(',', ''))
                if 0 <= v <= 99999:
                    return v
            except ValueError:
                pass

        explicit = re.findall(r'final\s+answer\s*(?:is|:)\s*([0-9,]+)', text, flags=re.IGNORECASE)
        if explicit:
            try:
                v = int(explicit[-1].replace(',', ''))
                if 0 <= v <= 99999:
                    return v
            except ValueError:
                pass

        return None

    def _compute_mean_entropy(self, top_logprobs: list) -> float:
        if not top_logprobs:
            return float('inf')

        total_entropy = 0.0
        token_count = 0

        for token_top in top_logprobs:
            entries = []
            if isinstance(token_top, dict):
                entries = list(token_top.values())
            elif isinstance(token_top, list):
                for item in token_top:
                    if isinstance(item, dict) and 'logprob' in item:
                        entries.append(item['logprob'])

            probs = [math.exp(lp) for lp in entries if isinstance(lp, (int, float))]
            s = sum(probs)
            if s <= 0:
                continue
            probs = [p / s for p in probs]
            entropy = -sum(p * math.log2(p) for p in probs if p > 0)
            total_entropy += entropy
            token_count += 1

        return total_entropy / token_count if token_count else float('inf')

    def _prepare_tool_code(self, code: str) -> str:
        stripped = code.strip()
        if not stripped:
            return code

        lines = stripped.split('\n')
        last_line = lines[-1].strip()
        if not last_line or last_line.startswith('#'):
            return stripped
        if 'print' in last_line or last_line.startswith('import '):
            return stripped

        try:
            ast.parse(last_line, mode='eval')
            lines[-1] = f'print({last_line})'
            return '\n'.join(lines)
        except Exception:
            return stripped

    def _execute_tool_calls(self, tool_calls: list, sandbox: AIMO3Sandbox):
        out_messages = []
        py_calls = 0
        py_errs = 0

        for tc in tool_calls:
            py_calls += 1
            fn_info = tc.get('function', {})
            fn_name = fn_info.get('name', '')
            raw_args = fn_info.get('arguments') or '{}'

            try:
                args = json.loads(raw_args)
            except json.JSONDecodeError:
                args = {'code': raw_args}

            if fn_name == 'python_math':
                code = self._prepare_tool_code(args.get('code', ''))
                output = sandbox.execute(code)
            else:
                output = f'[ERROR] Unknown tool: {fn_name}'

            if output.startswith('[ERROR]') or 'Traceback' in output or 'Error:' in output:
                py_errs += 1

            out_messages.append(
                {
                    'role': 'tool',
                    'tool_call_id': tc.get('id'),
                    'name': fn_name,
                    'content': output,
                }
            )

        return out_messages, py_calls, py_errs

    def _request_chat(self, payload: dict, timeout_seconds: float | None = None, stream: bool = False):
        effective_timeout = timeout_seconds if timeout_seconds is not None else self.cfg.session_timeout
        return requests.post(
            f'{self.service.base_url}/chat/completions',
            json=payload,
            timeout=effective_timeout,
            headers={'Content-Type': 'application/json'},
            stream=stream,
        )

    def _stream_chat_completion(self, payload: dict, timeout_seconds: float):
        # SSE stream parser for OpenAI-compatible /chat/completions.
        response = self._request_chat(payload, timeout_seconds=timeout_seconds, stream=True)

        if response.status_code != 200:
            text = response.text[:1000]
            response.close()
            raise RuntimeError(f'HTTP {response.status_code}: {text}')

        content_parts = []
        tool_calls_by_index = {}
        completion_chunks = 0

        try:
            for raw_line in response.iter_lines(decode_unicode=True):
                if not raw_line:
                    continue
                if not raw_line.startswith('data: '):
                    continue

                data = raw_line[6:]
                if data.strip() == '[DONE]':
                    break

                try:
                    obj = json.loads(data)
                except json.JSONDecodeError:
                    continue

                choices = obj.get('choices') or []
                if not choices:
                    continue

                delta = choices[0].get('delta') or {}

                txt = delta.get('content')
                if txt:
                    content_parts.append(txt)
                    completion_chunks += 1

                for tc in delta.get('tool_calls') or []:
                    idx = int(tc.get('index', 0))
                    entry = tool_calls_by_index.setdefault(
                        idx,
                        {
                            'id': None,
                            'type': 'function',
                            'function': {'name': '', 'arguments': ''},
                        },
                    )

                    if tc.get('id'):
                        entry['id'] = tc['id']

                    fn = tc.get('function') or {}
                    if fn.get('name'):
                        entry['function']['name'] = fn['name']
                    if fn.get('arguments'):
                        entry['function']['arguments'] += fn['arguments']

                # Early stop when boxed answer appears in stream.
                if content_parts:
                    tail = ''.join(content_parts[-self.cfg.search_tokens:])
                    if self._scan_for_answer(tail) is not None:
                        break

            content = ''.join(content_parts)
            tool_calls = [tool_calls_by_index[k] for k in sorted(tool_calls_by_index.keys())]

            return {
                'choices': [
                    {
                        'message': {
                            'role': 'assistant',
                            'content': content,
                            'tool_calls': tool_calls,
                        },
                        'logprobs': None,
                    }
                ],
                'usage': {
                    # Approximate token count from stream chunks when usage is absent.
                    'completion_tokens': max(0, completion_chunks),
                },
            }
        finally:
            response.close()

    def _estimate_message_tokens(self, messages: list) -> int:
        # Lightweight estimate used only for preflight trimming.
        total = 0
        for m in messages:
            content = m.get('content') or ''
            if isinstance(content, str):
                total += max(1, len(content) // 4)
            elif isinstance(content, list):
                total += max(1, len(str(content)) // 4)
            else:
                total += 8

            # Account for message framing and tool call payload overhead.
            total += 12
            if m.get('tool_calls'):
                total += max(1, len(json.dumps(m.get('tool_calls'))) // 4)

        return total

    def _trim_messages_for_context(self, messages: list) -> list:
        # Keep system prompt and the latest context; trim oldest intermediate turns.
        if not messages:
            return messages

        max_input = max(1, self.cfg.context_tokens - 128)
        trimmed = list(messages)

        while len(trimmed) > 2 and self._estimate_message_tokens(trimmed) > max_input:
            # Prefer dropping oldest non-system message.
            drop_idx = None
            for i in range(1, len(trimmed) - 1):
                if trimmed[i].get('role') != 'system':
                    drop_idx = i
                    break
            if drop_idx is None:
                break
            trimmed.pop(drop_idx)

        return trimmed

    def _looks_like_input_overflow(self, error_text: str) -> bool:
        lowered = (error_text or '').lower()
        return 'maximum context length' in lowered and 'input tokens' in lowered

    def _extract_allowed_completion_tokens(self, error_text: str) -> int | None:
        # Example error: request has 4013 input tokens (96 > 4096 - 4013)
        match = re.search(r'request has\s+(\d+)\s+input tokens', error_text)
        if not match:
            match = re.search(r'has\s+(\d+)\s+input tokens', error_text)
        if not match:
            return None

        input_tokens = int(match.group(1))
        # Keep a small safety margin to avoid boundary/off-by-one failures.
        allowed = self.cfg.context_tokens - input_tokens - 8
        if allowed <= 0:
            return 1
        return allowed

    def _chat_with_retries(self, messages: list, deadline: float):
        working_messages = self._trim_messages_for_context(messages)

        # 120B-style logic: dynamic max_tokens from remaining context with safety buffer.
        for _ in range(8):
            estimated_prompt_tokens = self._estimate_message_tokens(working_messages)
            remaining_tokens = int(self.cfg.context_tokens - estimated_prompt_tokens)

            if remaining_tokens < self.cfg.buffer_tokens:
                return None

            max_toks = max(1, remaining_tokens)

            remaining_time = deadline - time.time()
            if remaining_time <= 1:
                raise RuntimeError('Attempt deadline exceeded before API call')

            call_timeout = min(float(self.cfg.session_timeout), max(1.0, remaining_time))

            payload = {
                'model': self.cfg.served_model_name,
                'messages': working_messages,
                'tools': self.tools,
                'tool_choice': 'auto',
                'temperature': self.cfg.temperature,
                'max_tokens': max_toks,
                'stream': True,
            }

            try:
                return self._stream_chat_completion(payload, timeout_seconds=call_timeout)
            except RuntimeError as exc:
                err = str(exc)

                if 'HTTP 400' in err:
                    dynamic_toks = self._extract_allowed_completion_tokens(err)
                    if dynamic_toks is not None and dynamic_toks > 0:
                        fallback_payload = {
                            'model': self.cfg.served_model_name,
                            'messages': working_messages,
                            'tools': self.tools,
                            'tool_choice': 'auto',
                            'temperature': self.cfg.temperature,
                            'max_tokens': int(dynamic_toks),
                            'stream': True,
                        }
                        try:
                            return self._stream_chat_completion(fallback_payload, timeout_seconds=call_timeout)
                        except RuntimeError as fallback_exc:
                            err = str(fallback_exc)

                    if self._looks_like_input_overflow(err):
                        new_working_messages = self._trim_messages_for_context(working_messages)
                        if len(new_working_messages) < len(working_messages):
                            working_messages = new_working_messages
                            continue

                raise

        raise RuntimeError('Failed retries for /v1/chat/completions after context-adaptive attempts')

    def _process_attempt(self, problem: str, attempt_index: int, stop_event: threading.Event, deadline: float) -> dict:
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Answer': None,
                'Python Calls': 0,
                'Python Errors': 0,
                'Response Length': 0,
                'Entropy': float('inf'),
                'Raw': '[INFO] Skipped due to stop/deadline',
            }

        sandbox = None
        total_tokens = 0
        py_calls = 0
        py_errs = 0
        final_answer = None
        final_text = ''
        logprobs_buffer = []

        messages = [
            {'role': 'system', 'content': self.cfg.system_prompt},
            {'role': 'user', 'content': f'{problem} {self.cfg.preference_prompt}'},
        ]

        try:
            sandbox = self.sandbox_pool.acquire(timeout=self.cfg.sandbox_timeout)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                data = self._chat_with_retries(messages, deadline=deadline)
                if data is None:
                    break
                choice = data['choices'][0]
                msg = choice.get('message', {})
                content = msg.get('content') or ''
                tool_calls = msg.get('tool_calls') or []

                usage = data.get('usage') or {}
                total_tokens += int(usage.get('completion_tokens') or 0)

                lp = choice.get('logprobs')
                if isinstance(lp, dict):
                    for item in lp.get('content') or []:
                        top = item.get('top_logprobs')
                        if top:
                            logprobs_buffer.append(top)

                assistant_message = {'role': 'assistant', 'content': content}
                if tool_calls:
                    assistant_message['tool_calls'] = tool_calls
                messages.append(assistant_message)

                if content:
                    final_text = content
                    ans = self._scan_for_answer(content)
                    if ans is not None:
                        final_answer = ans
                        break

                if tool_calls:
                    tool_msgs, c, e = self._execute_tool_calls(tool_calls, sandbox)
                    py_calls += c
                    py_errs += e
                    messages.extend(tool_msgs)
                    continue

                messages.append(
                    {
                        'role': 'user',
                        'content': 'Now give only the final answer in the exact format \boxed{n}.',
                    }
                )

            if final_answer is None:
                assistant_texts = [m.get('content', '') for m in messages if m.get('role') == 'assistant' and m.get('content')]
                fallback_text = '\n'.join(assistant_texts[-4:])
                final_text = fallback_text
                final_answer = self._scan_for_answer(fallback_text)

        except Exception as exc:
            final_text = f'[ERROR] {exc}'
            py_errs += 1

        finally:
            if sandbox is not None:
                with contextlib.suppress(Exception):
                    sandbox.reset()
                with contextlib.suppress(Exception):
                    self.sandbox_pool.release(sandbox)

        return {
            'Attempt': attempt_index + 1,
            'Response Length': total_tokens,
            'Python Calls': py_calls,
            'Python Errors': py_errs,
            'Entropy': self._compute_mean_entropy(logprobs_buffer),
            'Answer': final_answer,
            'Raw': final_text,
        }

    def _select_answer(self, detailed_results: list[dict]) -> int:
        weights = defaultdict(float)
        votes = defaultdict(int)

        for r in detailed_results:
            ans = r.get('Answer')
            ent = r.get('Entropy', float('inf'))
            if ans is None:
                continue
            w = 1.0 / max(ent, 1e-9)
            weights[ans] += w
            votes[ans] += 1

        ranked = [{'Answer': a, 'Votes': votes[a], 'Score': weights[a]} for a in weights]
        ranked.sort(key=lambda x: x['Score'], reverse=True)

        if ranked:
            display(pd.DataFrame(ranked).round({'Score': 3}))
            final = int(ranked[0]['Answer'])
            print(f'\nFinal Answer: {final}\n')
            return final

        print('\nFinal Answer: 0\n')
        return 0

    def solve_problem(self, problem: str) -> tuple[int, list[dict]]:
        print(f'\nProblem: {problem}\n')

        elapsed = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed
        reserved = max(0, self.problems_remaining - 1) * self.cfg.base_problem_timeout

        budget = time_left - reserved
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        stop_event = threading.Event()
        detailed = []
        valid_answers = []

        with ThreadPoolExecutor(max_workers=min(self.cfg.workers, self.cfg.attempts)) as ex:
            futures = [ex.submit(self._process_attempt, problem, i, stop_event, deadline) for i in range(self.cfg.attempts)]

            for fut in as_completed(futures):
                try:
                    result = fut.result()
                except Exception as exc:
                    print(f'Attempt future failed: {exc}')
                    continue

                detailed.append(result)
                if result.get('Answer') is not None:
                    valid_answers.append(result['Answer'])

                top = Counter(valid_answers).most_common(1)
                if top and top[0][1] >= self.cfg.early_stop:
                    stop_event.set()
                    for f in futures:
                        f.cancel()
                    break

        self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed:
            df = pd.DataFrame(detailed)
            if 'Entropy' in df.columns:
                df['Entropy'] = df['Entropy'].round(3)
            if 'Answer' in df.columns:
                df['Answer'] = df['Answer'].astype('Int64')
            display(df[['Attempt', 'Answer', 'Python Calls', 'Python Errors', 'Response Length', 'Entropy']])

        if not valid_answers:
            print('\nResult: 0\n')
            return 0, detailed

        return self._select_answer(detailed), detailed

    def close(self):
        self.sandbox_pool.close()
        self.service.close()

    def __del__(self):
        self.close()


In [ ]:
def evaluate_reference_csv(
    solver: AIMO3Solver,
    reference_path='/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv',
    limit=None,
    print_raw_responses=True,
):
    rows = []
    with open(reference_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if limit is not None and i >= limit:
                break
            rows.append(row)

    total = len(rows)
    correct = 0
    results = []

    for i, row in enumerate(rows, 1):
        qid = row['id']
        problem = row['problem']
        expected = int(row['answer'])

        gc.disable()
        try:
            pred, detailed = solver.solve_problem(problem)
            raw_text = detailed[0].get('Raw', '') if detailed else ''
            tool_calls = sum(int(x.get('Python Calls', 0)) for x in detailed)
        except Exception as exc:
            pred, raw_text, tool_calls = None, f'[ERROR] {exc}', 0
            detailed = []
        finally:
            gc.enable()
            gc.collect()

        is_correct = pred == expected
        correct += int(is_correct)

        results.append(
            {
                'id': qid,
                'expected': expected,
                'predicted': pred,
                'correct': is_correct,
                'tool_calls': tool_calls,
                'raw': raw_text,
                'attempt_details': detailed,
            }
        )

        acc = correct / i
        print(f'[{i}/{total}] id={qid} expected={expected} pred={pred} tool_calls={tool_calls} correct={is_correct} running_acc={acc:.3f}')

        if print_raw_responses:
            print(f'--- Raw responses for id={qid} ---')
            if not detailed:
                print(raw_text)
            else:
                for attempt in sorted(detailed, key=lambda x: x.get('Attempt', 0)):
                    attempt_id = attempt.get('Attempt')
                    attempt_answer = attempt.get('Answer')
                    attempt_tools = attempt.get('Python Calls', 0)
                    response = attempt.get('Raw', '')
                    print(f'[Attempt {attempt_id}] parsed_answer={attempt_answer} python_calls={attempt_tools}')
                    print(response)
                    print('-' * 80)

    final_acc = correct / total if total else 0.0
    print(f'\nFinal accuracy: {correct}/{total} = {final_acc:.3f}')
    return results


In [ ]:
if 'solver' in globals():
    with contextlib.suppress(Exception):
        solver.close()

solver = AIMO3Solver(cfg)


def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    id_value = id_.item(0)
    question_text = question.item(0)

    gc.disable()
    try:
        final_answer, _ = solver.solve_problem(question_text)
    finally:
        gc.enable()
        gc.collect()

    return pl.DataFrame({'id': id_value, 'answer': final_answer})


# Quick smoke test
smoke_problem = 'What is 12*13? Return final answer in \\boxed{}'
smoke_pred, smoke_details = solver.solve_problem(smoke_problem)
print('Smoke test prediction:', smoke_pred)
if smoke_details:
    print(smoke_details[0].get('Raw', ''))


# Optional local evaluation examples:
# results = evaluate_reference_csv(solver, limit=10)
# results = evaluate_reference_csv(solver, limit=None)


if aimo3_inference is not None:
    inference_server = aimo3_inference.AIMO3InferenceServer(predict)

    if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        inference_server.serve()
    else:
        pass
        # inference_server.run_local_gateway(
        #     ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
        # )
